### 1. Run the nanoreactor 

In [1]:
from arcskin.nanoreactor_run import SMILESPack
from arcskin.nanoreactor_run import XTBNanoReactor


In [2]:
molecules = {"NO": "[N]=O", "O2": "O=O"}
molecules = {"N2":"N#N","O2": "O=O"}
molecules = {"SO2":"O=S=O","NO2":"N(=O)[O]"}
molecules = {"SO3":'O=S(=O)=O',"H2O":"O"}

In [13]:
sp = SMILESPack(molecules.values())
#sp = SMILESPack(["[N]=O","[N]=O"])
cells, charges = sp.run_multiple(
    4, counts=[4, 4], min_dist=2.2
)  # 4 of each molecule, with a minimum distance of 2\AA
cells,charges


([Atoms(symbols='OSO3SO3SO3SO3H2OH2OH2OH2', pbc=False),
  Atoms(symbols='OSO3SO3SO3SO3H2OH2OH2OH2', pbc=False),
  Atoms(symbols='OSO3SO3SO3SO3H2OH2OH2OH2', pbc=False),
  Atoms(symbols='OSO3SO3SO3SO3H2OH2OH2OH2', pbc=False)],
 [0, 0, 0, 0])

In [14]:
xnr = XTBNanoReactor(
    combination=tuple(molecules.values()),
    cells=cells,
    charges=charges,
    temperature=4000,
    time=5,
)
#xnr = XTBNanoReactor(combination = ("[N]=O","[N]=O"),cells=cells, charges=charges, time=5)
xnr.run(location="./example_run/")


In [15]:
xnr.trajectory_locations


['./example_run/seed_0/xtb.trj',
 './example_run/seed_1/xtb.trj',
 './example_run/seed_2/xtb.trj',
 './example_run/seed_3/xtb.trj']

### 2. Analyse the run 

In [ ]:
from arcskin.nanoreactor_analysis import AnalyseNanoReactor

anr = AnalyseNanoReactor(skip_step=20, reactants=list(molecules.values()))
#anr = AnalyseNanoReactor(skip_step=20, reactants=["[N]=O","[N]=O"])

anr.read_trajectory(xnr.trajectory_locations, format="xyz")
anr.analyse_trajectories()


calculating with default calculator: aimnet


 16%|█▌        | 8/50 [01:03<05:12,  7.44s/it][13:19:55] UFFTYPER: Unrecognized atom type: S_5+4 (1)
[13:19:55] UFFTYPER: Unrecognized atom type: S_5+4 (1)
 98%|█████████▊| 49/50 [07:11<00:12, 12.83s/it]

In [ ]:
import pandas as pd

df = pd.DataFrame(anr.unique_smiles_counter).sum(axis=1).sort_values(ascending=False)
df

O=S=O            17
O=O               8
O=S1OO1           6
[O]S1([O])OO1     1
dtype: int64

In [ ]:
anr.unique_smiles

['O=S=O', 'O=O', 'O=S1OO1', '[O]S1([O])OO1']

In [9]:
anr.get_sensible_smiles()
possible_paths = anr.get_possible_paths_dict()
possible_paths

[13:06:12] UFFTYPER: Unrecognized atom type: S_5+4 (1)


### 3. Calculate the Energy barriers with NEB 

In [ ]:
from arcskin.make_neb_endpoints import make_endpoints

reaction_endpoints = {}
for k,v in possible_paths.items():
    try:
        reactants, products = make_endpoints(v["reactants"],v["products"],shift=1.5)
        reaction_endpoints[k] = {
            "reactants":reactants,
            "products":products,
            "overall":possible_paths[k]
                                 }
    except Exception:
        print(f"{k} failed")
        pass

In [ ]:
reaction_endpoints

In [ ]:
from arcskin.run_neb import run_all

results = run_all(reaction_endpoints,n_images=8,fmax=0.05)

In [ ]:
results


In [ ]:
import seaborn as sns 
import matplotlib.pyplot as plt 
from ase.mep import NEBTools
from ase.utils import forcecurve
import seaborn as sns 
import matplotlib.pyplot as plt 
from ase.mep import NEBTools
from ase.utils import forcecurve
import collections 

sns.set_style('whitegrid')
sns.set_palette('pastel')

def plot_nebruns(image_dict,ax,**kws):
    for k,v in image_dict.items():
        fit = forcecurve.fit_images(v["images"])
        Ef = fit[1].max()
        ax.plot(fit[2],fit[3],label=f'{k}: E$_a$ = {Ef :.2F} eV, dE = {v["dE"]:.2F}eV ')
        ax.plot(fit[0],fit[1],**kws)

kws = {'marker':'o',
       'linestyle':'None',
       'color':'grey',
       'markerfacecolor':'white',
       'markersize':5}

fig,ax = plt.subplots(ncols=1,nrows=1,figsize=(5,5),dpi=300,sharey=True,sharex=True)
plot_nebruns(image_dict=results.T,ax=ax,**kws)
ax.set_ylabel('Energy (eV)')
ax.set_xlabel('Reaction Coordinate')
    #flat_ax[i].set_title(f"expansion={nebbys[i]["expansion"]},rattle={nebbys[i]["rattle"]},nimages={nebbys[i]["nimages"]}")
ax.legend(edgecolor="black")

In [ ]:
from ase.visualize import view 

view(results["images"][0],viewer="ngl")